# Can you build the growth team's Monday table alone, end to end?

**Week 2, Thursday. The escalated case, unguided, 50 minutes.** The brief is
`exercises/unguided/C2_W02_D04_escalated_case_STUDENT.md`, and this notebook is where its five
parts run. Each step carries lettered choices written as comments above a `__TODOn__`
placeholder: replace the placeholder with the code of the option you choose and run the step's
check. Run as shipped, the notebook stops at `__TODO1__` with a `NameError`, which is how it is
meant to start.

> **The client asks.** "One table, one row per customer, refreshed every Monday: how recently,
> how often, how much, the segment, whether the monsoon sale reached them, and the two flags we
> act on. And one view of it by month that we can put on a slide."
>
> The growth team, Kalpa Retail

**The data.** Kalpa Retail's warehouse holds 1,000 orders placed between April and September
2026, Q1 (April to June) and Q2 (July to September), worth Rs 19,84,00,000 at the prices
charged, and a customer list of 340 customers in four segments: Retail-Core and Retail-Plus,
the two consumer tiers, of which Retail-Plus is the paid membership; Student; and Business,
Kalpa's sales to companies. The campaign platform's feed lists the customers the monsoon sale
reached in August 2026, with the date it reached each one. Spend is the value of a customer's
orders at the prices charged, whatever became of each order afterwards.

**The growth team's rules.** One row per customer on the list. A customer the feed names more
than once was reached once, on the first date the feed gives. *Lapsed:* no order in the 60
days to the table's as-of date, which step 3 asks you to choose; a customer who never ordered
is not lapsed, since there is nothing to win back, and gets the first-order nudge instead. *Falling:* Wednesday's rule, spend lower in August than in July and lower again in
September than in August, each reading a real calendar month after the one before, so a month
with no order breaks the run.

> **Kavya's review.** "A table that runs is only a draft until every check under it has passed.
> Post your letters and the four numbers the last cell prints."

**Setup.** The next cell reads the orders and the customer list from the warehouse and the campaign platform's feed from the day's `data/` folder.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
import pandas as pd

ENG = kit.engine()                  # the Kalpa warehouse in Postgres, read only
orders = pd.read_sql(
    "SELECT order_id, customer_id, order_date, quarter, channel, amount, status FROM orders",
    ENG, parse_dates=["order_date"])
customers = pd.read_sql("SELECT customer_id, segment, city FROM customers", ENG)
print(f"{len(orders):,} orders and {len(customers):,} customers read from the warehouse")

exposure = pd.read_csv(kit.data_dir() / "C2_W02_D04_exposure_STUDENT.csv", parse_dates=["exposed_date"])
print("the campaign platform's exposure feed, columns:", list(exposure.columns))

In [ ]:
kit.side_by_side(
    kit.ladder(["every customer, three numbers", "the sale, one row each", "the two flags",
                "one view for a slide", "one run, with guards"], show=False),
    kit.vflow(["the warehouse and the feed", "one table, one row per customer", "a slide, and a CSV for Friday"],
              title="Five steps, each with its checks", show=False),
)

## Step 1. Does the table hold every customer, with the three numbers each?

Where it is used at work: every table a growth team acts on is a list of the people it acts on,
and a person missing from it gets nothing.

In [ ]:
rfm = (orders.groupby("customer_id")
             .agg(last_order=("order_date", "max"), frequency=("order_id", "count"),
                  spend=("amount", "sum"))
             .reset_index())

# TODO 1. Which frame should the table start from?
#   a) customers[customers["segment"] != "Business"]
#   b) rfm[["customer_id"]]
#   c) customers
#   d) exposure[["customer_id"]].drop_duplicates()
table = __TODO1__.merge(rfm, on="customer_id", how="left", validate="one_to_one")

# TODO 2. What goes in the frequency of a customer who never ordered?
#   a) table["frequency"]
#   b) table["frequency"].fillna(0).astype("int64")
#   c) table["frequency"].fillna(table["frequency"].median())
#   d) table["frequency"].dropna()
table = table.assign(frequency=__TODO2__, spend=table["spend"].fillna(0))
print(f"{len(table)} rows")

In [ ]:
in_list = kit.sql("SELECT count(*) AS n FROM customers")[0]["n"]
kit.check("one row for every customer on the list", len(table) == in_list, f"{len(table)} rows")
kit.check("spend adds back to the warehouse", table["spend"].sum() == orders["amount"].sum())
kit.check("every customer has a frequency, and the frequencies add up to the warehouse's orders",
          table["frequency"].notna().all() and table["frequency"].sum() == len(orders))

## Step 2. Which customers did the sale reach, one row each, and does a second count agree?

Where it is used at work: any feed from another team's system is checked against the promise
it makes before it is joined to a table other people act on.

In [ ]:
ordered = exposure.sort_values("exposed_date")
# TODO 3. Which call applies the growth team's rule: one row per customer, on the first date the feed gives?
#   a) ordered.drop_duplicates("customer_id", keep="first")
#   b) ordered.drop_duplicates("customer_id", keep=False)
#   c) ordered.drop_duplicates("customer_id", keep="last")
#   d) ordered.drop_duplicates(["customer_id", "exposed_date"])
first_touch = __TODO3__[["customer_id", "exposed_date"]]

# TODO 4. Which promise should the merge carry?
#   a) "one_to_many"
#   b) "many_to_many"
#   c) None
#   d) "one_to_one"
PROMISE = __TODO4__
table = table.merge(first_touch, on="customer_id", how="left", validate=PROMISE)
table = table.assign(reached=table["exposed_date"].notna())

# TODO 5. Which count, sharing no code with the rule or the merge, should equal the reached flags?
#   The count is a function, so the check can run it again on a table where the merge lost someone.
#   a) len(first_touch)
#   b) int(table["exposed_date"].notna().sum())
#   c) exposure["customer_id"].nunique()
#   d) len(exposure)
def second_count(exposure, first_touch, table):
    return __TODO5__
print(f"{len(table)} rows after the merge")

In [ ]:
kit.check("still one row per customer after the merge", len(table) == in_list and table["customer_id"].is_unique)
kit.check("spend did not move in the merge", table["spend"].sum() == orders["amount"].sum())
earliest = exposure.groupby("customer_id")["exposed_date"].min()
kit.check("each reached customer carries the first date the feed gives",
          (first_touch.set_index("customer_id")["exposed_date"].sort_index() == earliest.reindex(sorted(first_touch["customer_id"]))).all())
try:
    table[["customer_id"]].merge(exposure, on="customer_id", how="left", validate=PROMISE)
    raw_error = None
except Exception as error:                     # keep only the kind of error, never its message
    raw_error = type(error).__name__
kit.check("the promise you chose would have stopped the raw feed", raw_error == "MergeError")
kit.check("the second count agrees with the reached flags",
          int(second_count(exposure, first_touch, table)) == int(table["reached"].sum()))
lost = first_touch.iloc[1:]                                   # a merge that lost one reached customer
lost_table = table.assign(exposed_date=table["exposed_date"].where(table["customer_id"].isin(lost["customer_id"])))
lost_table = lost_table.assign(reached=lost_table["exposed_date"].notna())
kit.check("and it disagrees when the merge loses a reached customer",
          int(second_count(exposure, lost, lost_table)) != int(lost_table["reached"].sum()))
reach = table.groupby("segment")["reached"].sum().astype(int)
kit.columns(reach.index.tolist(), [("reached by the sale", reach.tolist())],
            title="Reached customers per segment, one row each")

## Step 3. Who has gone quiet, and whose monthly spend is falling?

Where it is used at work: a retention team's weekly list is two flags, one for customers who
stopped and one for customers who are slowing, and each has to mean the same thing every week.

In [ ]:
# TODO 6. Which date is the table's as-of date, the date recency is counted to?
#   a) pd.Timestamp.today().normalize()
#   b) pd.Timestamp("2026-09-30")
#   c) table["last_order"].max() + pd.Timedelta(days=1)
#   d) orders["order_date"].max()
AS_OF = __TODO6__
table = table.assign(as_of=AS_OF, recency_days=(AS_OF - table["last_order"]).dt.days)

# TODO 7. Which condition flags a lapsed customer?
#   a) table["recency_days"] > 60
#   b) table["frequency"] == 0
#   c) table["recency_days"].fillna(9999) > 60
#   d) (pd.Timestamp.today() - table["last_order"]).dt.days > 60
table = table.assign(lapsed=__TODO7__)

monthly = (orders.assign(month_num=orders["order_date"].dt.year * 12 + orders["order_date"].dt.month)
                 .groupby(["customer_id", "month_num"], as_index=False)["amount"].sum()
                 .rename(columns={"amount": "spend"})
                 .sort_values(["customer_id", "month_num"]))
# TODO 8. Which grouping gives each customer's earlier monthly readings, as Wednesday's LAG did?
#   a) monthly
#   b) monthly.groupby("customer_id")
#   c) monthly.groupby("month_num")
#   d) monthly.groupby(["customer_id", "month_num"])
by = __TODO8__
monthly = monthly.assign(spend_1=by["spend"].shift(1), spend_2=by["spend"].shift(2),
                         month_num_2=by["month_num"].shift(2))
SEPTEMBER = 2026 * 12 + 9

# TODO 9. Which test keeps a fall only when the two readings before September are July and August?
#   a) monthly["month_num"] - monthly["month_num_2"] >= 2
#   b) monthly["month_num_2"].notna()
#   c) monthly["month_num"] - monthly["month_num_2"] <= 3
#   d) monthly["month_num"] - monthly["month_num_2"] == 2
real_months = __TODO9__
falling = monthly[(monthly["month_num"] == SEPTEMBER) & real_months
                  & (monthly["spend"] < monthly["spend_1"]) & (monthly["spend_1"] < monthly["spend_2"])]
table = table.assign(falling=table["customer_id"].isin(falling["customer_id"]))
print("both flags are set")

In [ ]:
def refresh_query(n):
    """The n-th query in chapter 6's sql/C2_W02_D04_06_refresh_STUDENT.sql, the warehouse's own count."""
    text = (kit.data_dir().parent / "sql" / "C2_W02_D04_06_refresh_STUDENT.sql").read_text(encoding="utf-8")
    return [query for query in text.split(";") if query.strip()][n]

kit.check("the smallest recency is 0 days", table["recency_days"].min() == 0)
by_sql = int(kit.sql(refresh_query(1))[0]["win_back"])
kit.check("the win-back list matches the warehouse's own count", int(table["lapsed"].sum()) == by_sql)
wednesday = {r["customer_id"] for r in kit.sql(refresh_query(2).replace("SELECT count(*) AS falling", "SELECT customer_id"))}
kit.check("pandas flags exactly the customers Wednesday's calendar-checked LAG flags",
          set(table.loc[table["falling"], "customer_id"]) == wednesday)
kit.check("both flags are booleans", table["lapsed"].dtype == bool and table["falling"].dtype == bool)
lap = table.groupby("segment")["lapsed"].sum().astype(int)
kit.columns(lap.index.tolist(), [("lapsed, as of the table's as-of date", lap.tolist())],
            title="The win-back list per segment")

## Step 4. How does spend move month by month in each segment?

Where it is used at work: the slide a growth review opens on is a trend by segment, and the
trend is only as right as the number behind each point.

In [ ]:
seg_orders = (orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
                    .assign(month=orders["order_date"].dt.to_period("M").astype(str)))
# TODO 10. Which call builds the slide's view, months down the side and one column per segment?
#   a) seg_orders.pivot_table(index="segment", columns="month", values="amount", aggfunc="sum")
#   b) seg_orders.pivot_table(index="month", columns="segment", values="amount", aggfunc="sum")
#   c) seg_orders.pivot_table(index="month", columns="segment", values="amount")
#   d) seg_orders.pivot_table(index="month", columns="segment", values="amount", aggfunc="count")
view = __TODO10__
print("view shape:", view.shape)

In [ ]:
kit.check("the view adds back to the warehouse", view.values.sum() == orders["amount"].sum())
kit.check("one row per month, April to September", list(view.index) == sorted(seg_orders["month"].unique()))
kit.line(list(view.index), [(s, view[s].tolist(), "lit" if s == "Retail-Plus" else "")
                            for s in ["Retail-Core", "Retail-Plus", "Student"]], fmt=kit.rupees,
         title="The slide: consumer spend by month and segment")

## Step 5. Will the table rebuild itself next Monday and stop if something breaks?

Where it is used at work: a scheduled job that feeds a campaign is trusted only when it
refuses to write a table that fails its checks.

In [ ]:
def guard(t):
    # TODO 11. Which guard stops a table that has stopped being one row per customer?
    #   a) t["customer_id"].is_unique
    #   b) len(t) == t["customer_id"].count()
    #   c) t.duplicated().sum() == 0
    #   d) t["customer_id"].notna().all()
    assert __TODO11__, "the customer table is no longer one row per customer"
    assert len(t) == in_list, "the table no longer holds every customer on the list"
    assert t["spend"].sum() == orders["amount"].sum(), "spend no longer adds to the warehouse"
    # TODO 12. Which guard catches recency counted to the wrong date?
    #   a) t["recency_days"].max() <= 180
    #   b) t["recency_days"].notna().all()
    #   c) t["recency_days"].min() == 0
    #   d) t["as_of"].nunique() == 1
    assert __TODO12__, "recency was not counted to the table's as-of date"
    return t

run_1 = guard(table.copy())
run_2 = guard(table.copy())
print(len(run_1), "rows passed every guard")

In [ ]:
kit.check("two guarded runs give the same table", run_1.equals(run_2))
twice = table.copy()
twice.loc[twice.index[150], "customer_id"] = twice["customer_id"].iloc[151]   # one id written over another, rows unchanged
with kit.expect_error() as stopped:
    guard(twice)
kit.check("a customer id written twice stops the run, with the row count unchanged",
          stopped.name == "AssertionError" and "one row per customer" in stopped.message)
early = table.assign(recency_days=(pd.Timestamp("2026-09-21") - table["last_order"]).dt.days)
with kit.expect_error() as stopped_early:
    guard(early)                                              # counted to a week before the data ends
kit.check("recency counted to a date before the data ends stops the run",
          stopped_early.name == "AssertionError" and "as-of date" in stopped_early.message)
out = pathlib.Path("output"); out.mkdir(exist_ok=True)
run_1.to_csv(out / "C2_W02_D04_customer_table_STUDENT.csv", index=False)
kit.stats([(len(run_1), "customers", "one row each"), (kit.rupees(run_1["spend"].sum()), "spend", "adds to the warehouse"),
           (int(run_1["reached"].sum()), "reached by the sale", "once each"),
           (int(run_1["lapsed"].sum()), "on the win-back list", "as of the table's as-of date")])
kit.flow(["warehouse + feed", "the table, guarded", "output/ CSV\nfor Friday"], lit=1,
         title="The Monday table, one guarded run")

**The last question: what changes when the orders run to crores?** Next year the orders table
may hold 5 crore rows, too many to read into pandas every Monday, while the growth team still
wants the same three numbers for every customer who ordered. Step 1 would then read a
query's answer in place of the orders.

In [ ]:
# TODO 13. Which query should step 1 read instead, so the three numbers still arrive?
#   a) "SELECT customer_id, max(order_date) AS last_order, count(*) AS frequency, avg(amount) AS spend FROM orders GROUP BY customer_id"
#   b) "SELECT customer_id, max(order_date) AS last_order, count(*) AS frequency, sum(amount) AS spend FROM orders GROUP BY customer_id"
#   c) "SELECT customer_id, max(order_date) AS last_order, count(*) AS frequency, sum(amount) AS spend FROM orders WHERE order_date >= DATE '2026-07-01' GROUP BY customer_id"
#   d) "SELECT c.customer_id, max(o.order_date) AS last_order, count(*) AS frequency, sum(o.amount) AS spend FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id GROUP BY c.customer_id"
at_scale = pd.read_sql(__TODO13__, ENG)
print(len(at_scale), "rows came back from the warehouse")

In [ ]:
both = rfm.merge(at_scale, on="customer_id", how="outer", suffixes=("", "_sql"), indicator=True)
same = (len(at_scale) == len(rfm) and (both["_merge"] == "both").all()
        and {"last_order", "frequency", "spend"} <= set(at_scale.columns)
        and (both["frequency"] == both["frequency_sql"]).all()
        and (both["spend"] == both["spend_sql"].astype(float)).all()
        and (both["last_order"] == pd.to_datetime(both["last_order_sql"])).all())
kit.check("the warehouse's answer gives every ordering customer step 1's three numbers", same)
kit.check("it sends one row per customer who ordered, whatever the orders' size", len(at_scale) == len(rfm))
kit.check_summary()

**Post:** your thirteen letters in order, and the four numbers above. The CSV in `output/` is
what you bring to Friday.